# YOLOv8-Pose Training on FreiHAND (Google Drive Edition)
This notebook is configured to run from your Google Drive project folder.

**Structure:**
- Script location: `/content/drive/MyDrive/Colab Notebooks/CARMA/scripts`
- Data location: `/content/drive/MyDrive/Colab Notebooks/CARMA/data`

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Navigate to your project directory
import os
import sys
project_path = '/content/drive/MyDrive/Colab Notebooks/CARMA/scripts'
os.chdir(project_path)

# Add current directory to sys.path so we can import our modules
if project_path not in sys.path:
    sys.path.append(project_path)

print(f"Currently working in: {os.getcwd()}")

In [ ]:
# 3. Install Requirements
!pip install ultralytics
from ultralytics import YOLO
from convert_freihand import convert_to_yolo_pose

## 2. Unzip Dataset to Local Disk
**IMPORTANT:** We unzip from Drive to `/content/` because training directly on Google Drive is extremely slow due to I/O latency.

In [ ]:
zip_path = '../data/FreiHAND_pub_v2.zip'
local_extract_path = '/content/freihand_raw'

if os.path.exists(zip_path):
    !unzip -q "{zip_path}" -d "{local_extract_path}"
    print("Unzip to local disk complete.")
else:
    print(f"ERROR: Could not find zip at {zip_path}")

## 3. Conversion to YOLO Pose Format

In [ ]:
# Setup local YOLO structure for fast training
local_dataset_path = '/content/dataset'

# Use the modular conversion function from our scripts
convert_to_yolo_pose(
    root_path=local_extract_path, 
    output_path=local_dataset_path
)

print("Conversion complete.")

## 4. Train YOLOv8-Pose

In [ ]:
yaml_content = """
path: /content/dataset
train: train/images
val: train/images
kpt_shape: [21, 3]
names:
  0: hand
"""
with open('/content/hand_pose.yaml', 'w') as f: f.write(yaml_content)

model = YOLO('yolov8n-pose.pt')

# Save results back to Drive
drive_results_path = '/content/drive/MyDrive/Colab Notebooks/CARMA/models/hand_pose_training'
os.makedirs(drive_results_path, exist_ok=True)

model.train(
    data='/content/hand_pose.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    scale=0.9,       # Zoom for close-ups
    translate=0.2,   # Translation for cut-off hands
    mosaic=1.0,      # Combine views
    project=drive_results_path # Save weights directly to Drive
)